# 04 - TF-IDF baselines

Bag-of-words baselines establish the lower bound reported in Table 3 of the
paper. Models are trained on the training partition and evaluated on a held-out
validation split; the test partition is reserved for the final comparison.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC, SVC

from spam_detection.config import CLASSIFIER_TEST_PATH, CLASSIFIER_TRAIN_PATH, RESULTS_DIR
from spam_detection.evaluation import binary_metrics, save_table
from spam_detection.utils import set_seed

set_seed(42)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## Load the prepared corpus

In [ ]:
train = pd.read_csv(CLASSIFIER_TRAIN_PATH)
test = pd.read_csv(CLASSIFIER_TEST_PATH)

X_train, X_val, y_train, y_val = train_test_split(
    train["text"],
    train["label"],
    test_size=0.15,
    random_state=42,
    stratify=train["label"],
)
print(len(X_train), len(X_val), len(test))

## Train the baselines

In [ ]:
vectorizer_args = dict(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True,
)

models = {
    "TF-IDF + LogReg": make_pipeline(
        TfidfVectorizer(**vectorizer_args),
        LogisticRegression(max_iter=1000, class_weight="balanced"),
    ),
    "TF-IDF + LinearSVM": make_pipeline(
        TfidfVectorizer(**vectorizer_args),
        LinearSVC(class_weight="balanced"),
    ),
    "TF-IDF + SVM": make_pipeline(
        TfidfVectorizer(**vectorizer_args),
        SVC(C=1, kernel="rbf", class_weight="balanced", random_state=42),
    ),
    "TF-IDF + Naive Bayes": make_pipeline(
        TfidfVectorizer(**vectorizer_args),
        ComplementNB(),
    ),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predicted = model.predict(X_val)
    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X_val)[:, 1]
    else:
        scores = model.decision_function(X_val)
    metrics = binary_metrics(y_val, predicted, scores)
    rows.append({"Model": name, **{key: round(value, 4) for key, value in metrics.items()}})

results = pd.DataFrame(rows).sort_values("F1", ascending=False)
display(results)

## Final evaluation on the held-out test set

In [ ]:
test_rows = []
for name, model in models.items():
    model.fit(train["text"], train["label"])
    predicted = model.predict(test["text"])
    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(test["text"])[:, 1]
    else:
        scores = model.decision_function(test["text"])
    metrics = binary_metrics(test["label"], predicted, scores)
    test_rows.append({"Model": name, **{key: round(value, 4) for key, value in metrics.items()}})

test_results = pd.DataFrame(test_rows).sort_values("F1", ascending=False)
display(test_results)
save_table(test_results, RESULTS_DIR / "tables" / "baselines_tfidf.csv")